# Análisis exploratorio de **Viajes de Taxi** en **Chicago**

Este notebook forma parte de **Patrones Lab**, una serie de proyectos pensados para explorar datos reales con un enfoque claro, visual y reproducible. En este caso, el trabajo se centra en una base de **viajes de taxi en Chicago**, con el objetivo de ordenar los datos, revisar su calidad, construir variables útiles y empezar a detectar patrones en duración, distancia, tarifa y zonas de la ciudad.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

Apago los warnings para que la lectura del notebook quede más limpia.

In [2]:
import warnings
warnings.filterwarnings("ignore")

### Carpeta del proyecto
Detecto desde dónde corre el notebook y fijo la raíz del proyecto.

In [3]:
cwd = Path.cwd().resolve()

if cwd.name == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

print("Project dir:", PROJECT_DIR.name)
print("Working dir:", cwd.name)

Project dir: 2026-03_taxi-trip-chicago
Working dir: notebooks


Armo rutas de trabajo

In [4]:
DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_STAGING = PROJECT_DIR / "data" / "staging"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"

TRIPS_FILE = DATA_RAW / "Taxi_Trips_20260324.csv"

### Cargo la base
Leo el CSV y miro tamaño más primeras filas para ver con qué arranco.

In [5]:
df = pd.read_csv(TRIPS_FILE)
print("trips shape:", df.shape)
df.head()

trips shape: (14219363, 23)


,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,...,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
0,0000184e7cd53cee95af32eba49c44e4d20adcd8,f538e6b729d1aaad4230e9dcd9dc2fd9a168826ddadbd6...,01/19/2024 05:00:00 PM,01/19/2024 06:00:00 PM,4051.0,17.12,1.703198e+10,1.703132e+10,76.0,32.0,...,4.0,60.00,Credit Card,Flash Cab,41.979071,-87.903040,POINT (-87.9030396611 41.9790708201),41.884987,-87.620993,POINT (-87.6209929134 41.8849871918)
1,000072ee076c9038868e239ca54185eb43959db0,e51e2c30caec952b40b8329a68b498e18ce8a1f40fa75c...,01/28/2024 02:30:00 PM,01/28/2024 03:00:00 PM,1749.0,12.70,NaN,NaN,6.0,NaN,...,0.0,33.75,Cash,Flash Cab,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),NaN,NaN,NaN
2,000074019d598c2b1d6e77fbae79e40b0461a2fc,aeb280ef3be3e27e081eb6e76027615b0d40925b84d3eb...,01/05/2024 09:00:00 AM,01/05/2024 09:00:00 AM,517.0,3.39,NaN,NaN,6.0,8.0,...,1.0,14.69,Mobile,Taxicab Insurance Agency Llc,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),41.899602,-87.633308,POINT (-87.6333080367 41.899602111)
3,00007572c5f92e2ff067e6f838a5ad74e83665d3,7d21c2ca227db8f27dda96612bfe5520ab408fa9a462c8...,01/22/2024 08:45:00 AM,01/22/2024 09:30:00 AM,2050.0,15.06,NaN,NaN,76.0,NaN,...,5.5,56.56,Credit Card,Globe Taxi,41.980264,-87.913625,POINT (-87.913624596 41.9802643146),NaN,NaN,NaN
4,00007c3e7546e2c7d15168586943a9c22c3856cf,8ef1056519939d511d24008e394f83e925d2539d668a00...,01/18/2024 07:15:00 PM,01/18/2024 07:30:00 PM,1004.0,1.18,1.703184e+10,1.703184e+10,32.0,32.0,...,0.0,19.66,Mobile,5 Star Taxi,41.880994,-87.632746,POINT (-87.6327464887 41.8809944707),41.880994,-87.632746,POINT (-87.6327464887 41.8809944707)


### Primer vistazo a la estructura
Reviso tipos de datos, nulos y forma general de la base.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14219363 entries, 0 to 14219362
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     object 
 1   Taxi ID                     object 
 2   Trip Start Timestamp        object 
 3   Trip End Timestamp          object 
 4   Trip Seconds                float64
 5   Trip Miles                  float64
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Fare                        float64
 11  Tips                        float64
 12  Tolls                       float64
 13  Extras                      float64
 14  Trip Total                  float64
 15  Payment Type                object 
 16  Company                     object 
 17  Pickup Centroid Latitude    float64
 18  Pickup Centroid Longitude   float64
 19  Pickup Centroid Loc

Ordeno los nombres para que queden fáciles de usar.

In [7]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "", regex=False)
)

In [8]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'pickup_centroid_location',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
       'dropoff_centroid__location'],
      dtype='object')

Corrijo una columna que vino con un guion bajo de más.

In [9]:
df = df.rename(columns={
    "dropoff_centroid__location": "dropoff_centroid_location"
})

### Fechas al formato correcto
Convierto inicio y fin del viaje a fecha real para poder trabajarlos bien.

In [10]:
df["trip_start_timestamp"] = pd.to_datetime(
    df["trip_start_timestamp"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

df["trip_end_timestamp"] = pd.to_datetime(
    df["trip_end_timestamp"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

In [11]:
df[["trip_start_timestamp", "trip_end_timestamp"]].head()

,trip_start_timestamp,trip_end_timestamp
0,2024-01-19 17:00:00,2024-01-19 18:00:00
1,2024-01-28 14:30:00,2024-01-28 15:00:00
2,2024-01-05 09:00:00,2024-01-05 09:00:00
3,2024-01-22 08:45:00,2024-01-22 09:30:00
4,2024-01-18 19:15:00,2024-01-18 19:30:00


Aseguro que montos, distancias y coordenadas queden en formato numérico.

In [12]:
cols_num = [
    "trip_seconds",
    "trip_miles",
    "fare",
    "tips",
    "tolls",
    "extras",
    "trip_total",
    "pickup_centroid_latitude",
    "pickup_centroid_longitude",
    "dropoff_centroid_latitude",
    "dropoff_centroid_longitude"
]

for c in cols_num:
    df[c] = pd.to_numeric(df[c], errors="coerce")

Hago lo mismo con tractos y community areas para dejarlos consistentes.

In [13]:
cols_id_geo = [
    "pickup_census_tract",
    "dropoff_census_tract",
    "pickup_community_area",
    "dropoff_community_area"
]

for c in cols_id_geo:
    df[c] = pd.to_numeric(df[c], errors="coerce")

Vuelvo a mirar la estructura para validar los cambios hechos hasta acá.

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14219363 entries, 0 to 14219362
Data columns (total 23 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   trip_id                     object        
 1   taxi_id                     object        
 2   trip_start_timestamp        datetime64[ns]
 3   trip_end_timestamp          datetime64[ns]
 4   trip_seconds                float64       
 5   trip_miles                  float64       
 6   pickup_census_tract         float64       
 7   dropoff_census_tract        float64       
 8   pickup_community_area       float64       
 9   dropoff_community_area      float64       
 10  fare                        float64       
 11  tips                        float64       
 12  tolls                       float64       
 13  extras                      float64       
 14  trip_total                  float64       
 15  payment_type                object        
 16  company         

Reviso unas filas para ver cómo quedó el dataset después de la limpieza inicial.

In [15]:
df.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location
0,0000184e7cd53cee95af32eba49c44e4d20adcd8,f538e6b729d1aaad4230e9dcd9dc2fd9a168826ddadbd6...,2024-01-19 17:00:00,2024-01-19 18:00:00,4051.0,17.12,1.703198e+10,1.703132e+10,76.0,32.0,...,4.0,60.00,Credit Card,Flash Cab,41.979071,-87.903040,POINT (-87.9030396611 41.9790708201),41.884987,-87.620993,POINT (-87.6209929134 41.8849871918)
1,000072ee076c9038868e239ca54185eb43959db0,e51e2c30caec952b40b8329a68b498e18ce8a1f40fa75c...,2024-01-28 14:30:00,2024-01-28 15:00:00,1749.0,12.70,NaN,NaN,6.0,NaN,...,0.0,33.75,Cash,Flash Cab,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),NaN,NaN,NaN
2,000074019d598c2b1d6e77fbae79e40b0461a2fc,aeb280ef3be3e27e081eb6e76027615b0d40925b84d3eb...,2024-01-05 09:00:00,2024-01-05 09:00:00,517.0,3.39,NaN,NaN,6.0,8.0,...,1.0,14.69,Mobile,Taxicab Insurance Agency Llc,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),41.899602,-87.633308,POINT (-87.6333080367 41.899602111)
3,00007572c5f92e2ff067e6f838a5ad74e83665d3,7d21c2ca227db8f27dda96612bfe5520ab408fa9a462c8...,2024-01-22 08:45:00,2024-01-22 09:30:00,2050.0,15.06,NaN,NaN,76.0,NaN,...,5.5,56.56,Credit Card,Globe Taxi,41.980264,-87.913625,POINT (-87.913624596 41.9802643146),NaN,NaN,NaN
4,00007c3e7546e2c7d15168586943a9c22c3856cf,8ef1056519939d511d24008e394f83e925d2539d668a00...,2024-01-18 19:15:00,2024-01-18 19:30:00,1004.0,1.18,1.703184e+10,1.703184e+10,32.0,32.0,...,0.0,19.66,Mobile,5 Star Taxi,41.880994,-87.632746,POINT (-87.6327464887 41.8809944707),41.880994,-87.632746,POINT (-87.6327464887 41.8809944707)


Veo en qué columnas faltan más datos.

In [16]:
df.isna().sum().sort_values(ascending=False)

dropoff_census_tract          8113518
pickup_census_tract           7922844
dropoff_community_area        1272573
dropoff_centroid_longitude    1196124
dropoff_centroid_location     1196124
dropoff_centroid_latitude     1196124
pickup_community_area          397311
pickup_centroid_location       390023
pickup_centroid_latitude       390023
pickup_centroid_longitude      390023
trip_total                      30499
fare                            30499
extras                          30499
tolls                           30499
tips                            30499
trip_seconds                     2649
trip_end_timestamp                189
trip_miles                        119
taxi_id                            11
trip_id                             0
trip_start_timestamp                0
payment_type                        0
company                             0
dtype: int64

Paso el mismo chequeo a porcentaje para compararlo mejor.

In [17]:
(df.isna().mean() * 100).sort_values(ascending=False).round(2)

dropoff_census_tract          57.06
pickup_census_tract           55.72
dropoff_community_area         8.95
dropoff_centroid_longitude     8.41
dropoff_centroid_location      8.41
dropoff_centroid_latitude      8.41
pickup_community_area          2.79
pickup_centroid_location       2.74
pickup_centroid_latitude       2.74
pickup_centroid_longitude      2.74
trip_total                     0.21
fare                           0.21
extras                         0.21
tolls                          0.21
tips                           0.21
trip_seconds                   0.02
trip_end_timestamp             0.00
trip_miles                     0.00
taxi_id                        0.00
trip_id                        0.00
trip_start_timestamp           0.00
payment_type                   0.00
company                        0.00
dtype: float64

### Creo variables de fecha
Saco año, mes, día, hora y fin de semana a partir del inicio del viaje.

In [18]:
df["trip_date"] = df["trip_start_timestamp"].dt.date
df["trip_year"] = df["trip_start_timestamp"].dt.year
df["trip_month"] = df["trip_start_timestamp"].dt.month
df["trip_day"] = df["trip_start_timestamp"].dt.day
df["trip_hour"] = df["trip_start_timestamp"].dt.hour
df["trip_weekday"] = df["trip_start_timestamp"].dt.day_name()
df["trip_weekday_num"] = df["trip_start_timestamp"].dt.weekday

df["is_weekend"] = df["trip_weekday_num"].isin([5, 6])

In [19]:
df[["trip_start_timestamp", "trip_date", "trip_year", "trip_month", "trip_hour", "trip_weekday", "trip_weekday_num"]].head()

,trip_start_timestamp,trip_date,trip_year,trip_month,trip_hour,trip_weekday,trip_weekday_num
0,2024-01-19 17:00:00,2024-01-19,2024,1,17,Friday,4
1,2024-01-28 14:30:00,2024-01-28,2024,1,14,Sunday,6
2,2024-01-05 09:00:00,2024-01-05,2024,1,9,Friday,4
3,2024-01-22 08:45:00,2024-01-22,2024,1,8,Monday,0
4,2024-01-18 19:15:00,2024-01-18,2024,1,19,Thursday,3


Miro estadísticos simples de tiempo, distancia y montos.

In [20]:
df[["trip_seconds", "trip_miles", "fare", "tips", "tolls", "extras", "trip_total"]].describe()

,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total
count,1.421671e+07,1.421924e+07,1.418886e+07,1.418886e+07,1.418886e+07,1.418886e+07,1.418886e+07
mean,1.225905e+03,6.567188e+00,2.246990e+01,2.770886e+00,3.059485e-02,2.033563e+00,2.754610e+01
std,1.610112e+03,7.600035e+00,3.317939e+01,4.236935e+00,3.815459e+00,8.786831e+00,3.744949e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,4.800000e+02,1.070000e+00,8.500000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.025000e+01
50%,9.000000e+02,3.060000e+00,1.500000e+01,0.000000e+00,0.000000e+00,0.000000e+00,1.778000e+01
75%,1.663000e+03,1.165000e+01,3.375000e+01,4.000000e+00,0.000000e+00,1.500000e+00,4.120000e+01
max,8.640000e+04,3.397800e+03,9.999750e+03,7.507500e+02,5.550000e+03,5.559500e+03,9.999750e+03


Chequeo desde qué fecha hasta qué fecha llegan los viajes.

In [21]:
df["trip_start_timestamp"].min(), df["trip_start_timestamp"].max()

(Timestamp('2024-01-01 00:00:00'), Timestamp('2026-03-01 00:00:00'))

Marco si cada viaje tiene monto y algo de información geográfica en origen y destino.

In [22]:
df["flag_monto"] = df["fare"].notna()

df["flag_pickup_geo"] = (
    df["pickup_community_area"].notna() |
    df["pickup_centroid_latitude"].notna() |
    df["pickup_centroid_longitude"].notna()
)

df["flag_dropoff_geo"] = (
    df["dropoff_community_area"].notna() |
    df["dropoff_centroid_latitude"].notna() |
    df["dropoff_centroid_longitude"].notna()
)

In [23]:
df[["flag_monto", "flag_pickup_geo", "flag_dropoff_geo"]].head()

,flag_monto,flag_pickup_geo,flag_dropoff_geo
0,True,True,True
1,True,True,False
2,True,True,True
3,True,True,False
4,True,True,True


### Paso duración a minutos y horas
Creo versiones más cómodas de la duración del viaje.

In [24]:
df["trip_minutes"] = df["trip_seconds"] / 60
df["trip_hours"] = df["trip_seconds"] / 3600

### Recalcular el total por partes
Sumo tarifa, propina, peajes y extras para tener un total armado por separado.

In [25]:
df["total_calculado"] = df["fare"] + df["tips"] + df["tolls"] + df["extras"]

In [26]:
#   df["diff_total"] = df["trip_total"] - df["total_calculado"]

In [27]:
# df["flag_total_consistente"] = df["diff_total"].abs() <= 0.01

In [28]:
# df["flag_total_consistente"].mean()

### Calculo velocidad estimada
Armo una velocidad simple usando distancia y duración.

In [29]:
df["speed_mph"] = df["trip_miles"] / df["trip_hours"]

### Calculo tarifa por milla
Llevo la tarifa a una métrica más comparable entre viajes.

In [30]:
df["fare_per_mile"] = df["fare"] / df["trip_miles"]

### Calculo total por milla
Hago lo mismo, pero usando el total completo del viaje.

In [31]:
df["trip_total_per_mile"] = df["trip_total"] / df["trip_miles"]

### Calculo tarifa por minuto
Paso la tarifa a una lógica por tiempo.

In [32]:
df["fare_per_minute"] = df["fare"] / df["trip_minutes"]

### Calculo total por minuto
Completo la familia de ratios con el total por minuto.

In [33]:
df["trip_total_per_minute"] = df["trip_total"] / df["trip_minutes"]

### Limpio infinitos en ratios
Cambio valores infinitos por nulos para no arrastrar errores raros.

In [34]:
import numpy as np

cols_ratio = [
    "speed_mph",
    "fare_per_mile",
    "trip_total_per_mile",
    "fare_per_minute",
    "trip_total_per_minute"
]

for c in cols_ratio:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan)

### Calculo porcentaje de propina
Mido qué parte de la tarifa representa la propina.

In [35]:
df["tip_pct"] = (df["tips"] / df["fare"]) * 100
df["tip_pct"] = df["tip_pct"].replace([np.inf, -np.inf], np.nan)

Señalo los registros que tienen las piezas básicas para análisis operativo.

In [36]:
df["flag_trip_core"] = (
    df["trip_seconds"].notna() &
    df["trip_miles"].notna() &
    df["trip_start_timestamp"].notna()
)

### Agrupo por franja horaria
Paso la hora a una banda simple para leer mejor los viajes a lo largo del día.

In [37]:
df["time_band"] = "night"

df.loc[df["trip_hour"].between(6, 11), "time_band"] = "morning"
df.loc[df["trip_hour"].between(12, 17), "time_band"] = "afternoon"
df.loc[df["trip_hour"].between(18, 23), "time_band"] = "evening"

In [38]:
df[
    [
        "trip_year", "trip_month", "trip_hour", "is_weekend", "time_band",
        "trip_minutes", "trip_hours", "total_calculado", "speed_mph", "fare_per_mile",
        "trip_total_per_mile", "fare_per_minute", "trip_total_per_minute",
        "tip_pct", "flag_trip_core"
    ]
].head()

,trip_year,trip_month,trip_hour,is_weekend,time_band,trip_minutes,trip_hours,total_calculado,speed_mph,fare_per_mile,trip_total_per_mile,fare_per_minute,trip_total_per_minute,tip_pct,flag_trip_core
0,2024,1,17,False,afternoon,67.516667,1.125278,59.50,15.214021,2.657710,3.504673,0.673908,0.888669,21.978022,True
1,2024,1,14,True,afternoon,29.150000,0.485833,33.75,26.140652,2.657480,2.657480,1.157804,1.157804,0.000000,True
2,2024,1,9,False,morning,8.616667,0.143611,14.69,23.605416,3.218289,4.333333,1.266151,1.704836,25.481210,True
3,2024,1,8,False,morning,34.166667,0.569444,56.06,26.446829,2.606242,3.755644,1.148780,1.655415,28.815287,True
4,2024,1,19,False,evening,16.733333,0.278889,19.66,4.231076,13.508475,16.661017,0.952590,1.174900,23.337516,True


In [40]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'pickup_centroid_location',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
       'dropoff_centroid_location', 'trip_date', 'trip_year', 'trip_month',
       'trip_day', 'trip_hour', 'trip_weekday', 'trip_weekday_num',
       'is_weekend', 'flag_monto', 'flag_pickup_geo', 'flag_dropoff_geo',
       'trip_minutes', 'trip_hours', 'total_calculado', 'speed_mph',
       'fare_per_mile', 'trip_total_per_mile', 'fare_per_minute',
       'trip_total_per_minute', 'tip_pct', 'flag_trip_core', 'time_band'],
      dtype='object')

### Ajusto casos imposibles
Pongo en nulo los ratios que no tienen sentido por divisiones o valores no válidos.

In [41]:
df.loc[df["trip_hours"] <= 0, "speed_mph"] = np.nan

df.loc[df["trip_miles"] <= 0, "fare_per_mile"] = np.nan
df.loc[df["trip_miles"] <= 0, "trip_total_per_mile"] = np.nan

df.loc[df["trip_minutes"] <= 0, "fare_per_minute"] = np.nan
df.loc[df["trip_minutes"] <= 0, "trip_total_per_minute"] = np.nan

df.loc[df["fare"] <= 0, "tip_pct"] = np.nan

Dejo juntas las variables numéricas que quiero mirar con más detalle.

In [42]:
cols = [
    "trip_seconds",
    "trip_miles",
    "fare",
    "tips",
    "tolls",
    "extras",
    "trip_total",
    "trip_minutes",
    "trip_hours",
    "speed_mph",
    "fare_per_mile",
    "trip_total_per_mile",
    "fare_per_minute",
    "trip_total_per_minute",
    "tip_pct"
]

#   df[cols].agg(["min", "max"]).T

In [43]:
df[cols].describe(percentiles=[0.01, 0.02, 0.05, 0.50, 0.95, 0.98, 0.99, 0.998, 0.999]).round(2).T

,count,mean,std,min,1%,2%,5%,50%,95%,98%,99%,99.8%,99.9%,max
trip_seconds,14216714.0,1225.91,1610.11,0.0,0.00,4.00,27.00,900.00,3195.00,3965.00,4581.00,7123.00,10654.29,86400.00
trip_miles,14219244.0,6.57,7.60,0.0,0.00,0.00,0.00,3.06,18.32,22.09,26.77,36.96,42.56,3397.80
fare,14188864.0,22.47,33.18,0.0,3.25,3.25,4.50,15.00,52.25,65.25,75.00,105.25,135.50,9999.75
tips,14188864.0,2.77,4.24,0.0,0.00,0.00,0.00,0.00,11.10,13.90,16.00,23.45,28.05,750.75
tolls,14188864.0,0.03,3.82,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,4.00,5.00,5550.00
extras,14188864.0,2.03,8.79,0.0,0.00,0.00,0.00,0.00,6.00,24.00,32.29,55.00,63.50,5559.50
trip_total,14188864.0,27.55,37.45,0.0,3.25,3.25,5.04,17.78,68.25,85.00,99.90,150.08,183.00,9999.75
trip_minutes,14216714.0,20.43,26.84,0.0,0.00,0.07,0.45,15.00,53.25,66.08,76.35,118.72,177.57,1440.00
trip_hours,14216714.0,0.34,0.45,0.0,0.00,0.00,0.01,0.25,0.89,1.10,1.27,1.98,2.96,24.00
speed_mph,14021457.0,17.59,174.71,0.0,0.00,0.00,0.00,13.38,40.67,46.88,50.60,58.88,63.53,423304.36


### Aplico filtros duros
Me quedo solo con viajes con duración, distancia y tarifa mínimas razonables.

- `trip_seconds >= 90`: elimina viajes demasiado breves y limpia registros cercanos a 0 segundos.
- `trip_seconds <= 5400`: recorta duraciones extremas (hasta 24 horas antes del filtro).

- `trip_miles >= 0.30`: elimina recorridos que suelen responder a errores, anulaciones o viajes no representativos.
- `trip_miles <= 50`: recorta distancias incompatibles con el uso habitual del taxi en la ciudad.

- `speed_mph >= 1`: excluye velocidades medias prácticamente nulas.
- `speed_mph <= 70`: elimina velocidades medias excesivas para una operación urbana.

- `fare >= 3.25`: respeta el piso mínimo tarifario y evita importes incompatibles.
- `fare <= 150`: recorta tarifas extremas sin afectar el centro de la distribución.

- `fare_per_mile >= 2`: fija un piso mínimo tarifario por milla y elimina cobros demasiado bajos para la distancia recorrida.
- `fare_per_mile <= 50`: recorta una cola alta distorsionada y evita ratios por milla incompatibles con un viaje normal.

- `fare_per_minute >= 0.3`: fija un piso mínimo ad tarifario por minuto y elimina viajes con cobro anormalmente bajo para su duración.
- `fare_per_minute <= 5`: elimina una cola alta extrema y deja el ratio en un rango compatible con viajes reales.

- `tolls <= 5`: limita peajes a valores dentro del alcance geográfico ya filtrado.
- `tips <= 30`: recorta propinas extremas que inflan el valor del viaje.
- `extras <= 50`: acota cargos adicionales muy altos.
- `tip_pct <= 100`: elimina propinas superiores al valor del fare, poco representativas para un análisis.

- `trip_total <= 190`: limita el monto total del viaje a un rango coherente con el tope definido para `fare` y cargos adicionales razonables.

- `trip_total_per_mile <= 60`: recorta la cola alta del total por milla.
- `trip_total_per_minute <= 6`: recorta la cola alta del total por minuto.

In [ ]:
# FILTROS DUROS

df = df[
    (df["trip_seconds"] <= 5400) &
    (df["trip_seconds"] >= 90) &   

    (df["trip_miles"] <= 50) &
    (df["trip_miles"] >= 0.30) &

    (df["speed_mph"] >= 1) &
    (df["speed_mph"] <= 70) &

    (df["fare"] <= 150) &
    (df["fare"] >= 3.25) &

    (df["fare_per_mile"] <= 50) &
    (df["fare_per_mile"] >= 2) &

    (df["fare_per_minute"] <= 5) &
    (df["fare_per_minute"] >= 0.3) &

    (df["tolls"] <= 5) &
    (df["tips"] <= 30) &
    (df["extras"] <= 50) &
    (df["tip_pct"] <= 100 ) &
    (df["trip_total"] <= 190 ) &

    (df["trip_total_per_mile"] <= 60 ) &

    (df["trip_total_per_minute"] <= 6 )

].copy()

Busco duplicados por trip_id

In [45]:
df["trip_id"].duplicated().sum()

np.int64(0)

Busco duplicados por combinación clave

In [46]:
df.duplicated(subset=["taxi_id", "trip_start_timestamp", "trip_end_timestamp"]).sum()

np.int64(36137)

Dejo juntas las variables numéricas que quiero mirar con más detalle.

In [47]:
#   cols = [
#       "trip_seconds",
#       "trip_miles",
#       "fare",
#       "tips",
#       "tolls",
#       "extras",
#       "trip_total",
#       "trip_minutes",
#       "trip_hours",
#       "speed_mph",
#       "fare_per_mile",
#       "trip_total_per_mile",
#       "fare_per_minute",
#       "trip_total_per_minute",
#       "tip_pct"
#   ]
#   
#   #   df[cols].agg(["min", "max"]).T

Vuelvo a analizar cómo quedaron las variables numéricas luego de los filtros aplicados

In [48]:
df[cols].describe(percentiles=[0.01, 0.02, 0.05, 0.50, 0.95, 0.98, 0.99, 0.998, 0.999]).round(2).T

,count,mean,std,min,1%,2%,5%,50%,95%,98%,99%,99.8%,99.9%,max
trip_seconds,12406255.0,1261.47,929.02,90.00,177.00,205.00,275.00,985.00,3178.00,3831.00,4260.00,4990.00,5166.00,5400.0
trip_miles,12406255.0,7.25,6.77,0.30,0.40,0.50,0.64,4.22,18.40,21.77,26.13,32.60,35.68,50.0
fare,12406255.0,22.20,16.17,3.25,4.50,5.00,5.50,15.75,49.25,61.10,68.45,85.19,93.75,150.0
tips,12406255.0,2.76,3.98,0.00,0.00,0.00,0.00,0.00,10.75,13.25,15.10,20.40,23.20,30.0
tolls,12406255.0,0.02,0.22,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,3.00,4.00,5.0
extras,12406255.0,1.86,4.82,0.00,0.00,0.00,0.00,0.00,6.00,21.00,28.95,40.25,44.00,50.0
trip_total,12406255.0,27.10,21.38,3.25,5.00,5.25,6.25,18.25,65.00,80.50,92.00,122.48,133.50,189.5
trip_minutes,12406255.0,21.02,15.48,1.50,2.95,3.42,4.58,16.42,52.97,63.85,71.00,83.17,86.10,90.0
trip_hours,12406255.0,0.35,0.26,0.02,0.05,0.06,0.08,0.27,0.88,1.06,1.18,1.39,1.44,1.5
speed_mph,12406255.0,18.38,11.21,1.00,3.73,4.71,6.00,14.97,41.13,46.96,50.41,57.13,59.78,70.0


Inspecciono los casos más largos en distancia para ver si suenan razonables.

In [49]:
df.sort_values("trip_miles", ascending=False)[
    ["trip_id", "trip_start_timestamp", "trip_seconds", "trip_miles", "fare", "trip_total", "payment_type", "company"]
].head(20)

,trip_id,trip_start_timestamp,trip_seconds,trip_miles,fare,trip_total,payment_type,company
7473261,3e8464bfcb8a73597756defac385e01476586d98,2025-04-02 01:15:00,4718.0,50.00,117.75,117.75,Cash,Sun Taxi
6585595,2ff00563d42ed1ca4380c1e9af3f16385cdb78cf,2025-01-17 04:00:00,3600.0,50.00,119.25,119.25,Cash,"Taxicab Insurance Agency, LLC"
9408801,6ac3034b3f9a8f457d57ec47c38a68734562f68c,2025-06-19 12:45:00,5004.0,50.00,119.75,129.75,Cash,Sun Taxi
7262812,c0640362c6e9fa907a33cfbac97608388d63ba28,2025-02-21 04:30:00,3060.0,50.00,117.00,117.00,Cash,"Taxicab Insurance Agency, LLC"
7121026,72752b02aac1722afe86b0f8c56a0f0557e476c7,2025-02-24 05:15:00,4320.0,50.00,125.00,125.00,Cash,"Taxicab Insurance Agency, LLC"
5809923,a05c45eadaaa87f163b837193748b49f29ef309a,2024-11-07 05:00:00,3000.0,50.00,117.00,117.00,Cash,"Taxicab Insurance Agency, LLC"
11315501,95d3751d8a43af6d82b0acbd91f32c94f0cc16c2,2025-09-28 12:30:00,3335.0,49.99,116.50,125.50,Credit Card,Taxicab Insurance Agency Llc
3152243,f89bacb860fb4a3c8da6cabde9ba13abc75d623d,2024-05-31 18:30:00,3421.0,49.98,116.25,151.50,Credit Card,5 Star Taxi
12667009,dd42abbb72af13b2d1229ce22da12df6c88bafa8,2025-11-14 08:00:00,3396.0,49.93,117.00,122.50,Credit Card,Flash Cab
3417277,38708a0d3a034c0ad429748a679310fb5a21f3ca,2024-07-22 04:30:00,4615.0,49.93,120.25,134.75,Credit Card,Medallion Leasin


Hago la misma revisión, pero con las duraciones más grandes.

In [50]:
df.sort_values("trip_seconds", ascending=False)[
    ["trip_id", "trip_start_timestamp", "trip_seconds", "trip_miles", "fare", "trip_total", "payment_type", "company"]
].head(20)

,trip_id,trip_start_timestamp,trip_seconds,trip_miles,fare,trip_total,payment_type,company
9444881,78a6f90926165525961ca79db380144e19dc03ee,2025-05-31 19:45:00,5400.0,20.50,58.00,74.50,Credit Card,Taxi Affiliation Services
4848485,fc987c684e723886b913468e982bc10eee14de7a,2024-09-04 08:00:00,5400.0,17.80,53.00,68.50,Credit Card,Taxi Affiliation Services
1312097,c75092a7166cfb514c678d00c8dd8e5e710bee38,2024-03-12 08:00:00,5400.0,14.09,48.50,62.00,Credit Card,Sun Taxi
7081947,5cdb0278994c93b0277d7406cb6edfaa985606a0,2025-02-14 19:00:00,5400.0,13.90,47.00,60.50,Credit Card,Chicago City Taxi Association
7671652,969214c891c1a4a06c65e0cecef39590be74134f,2025-04-25 14:15:00,5400.0,20.20,54.50,59.50,Cash,Taxi Affiliation Services
9984838,578f411dfa6cf28b099d97d2342af5e5537f3829,2025-07-01 15:45:00,5400.0,21.30,61.25,82.90,Credit Card,Taxi Affiliation Services
10544917,4d5200ba77fc271af3757df519a7b88646c188dc,2025-08-11 18:30:00,5400.0,17.50,48.00,62.50,Credit Card,Transit Administrative Center Inc
12017217,b1f49c364d5b0514fcc791fca117d9f58d1c09df,2025-10-10 16:00:00,5400.0,17.10,48.50,59.50,Credit Card,Transit Administrative Center Inc
1671714,7180a4f3e9d5cefeae6aac9eb087d6c7051f9cc9,2024-04-16 08:00:00,5400.0,16.80,52.00,56.00,Cash,Taxi Affiliation Services
9339885,4f8282a6e4146088b606733df86c071df67093e1,2025-06-06 20:00:00,5400.0,24.00,63.75,68.75,Cash,Taxi Affiliation Services


In [51]:
cols_auditoria = [
    "trip_seconds",
    "trip_miles",
    "fare",
    "tips",
    "tolls",
    "extras",
    "trip_total",
    "trip_minutes",
    "trip_hours",
    "speed_mph",
    "fare_per_mile",
    "trip_total_per_mile",
    "fare_per_minute",
    "trip_total_per_minute",
    "tip_pct"
]

cols_auditoria = [c for c in cols_auditoria if c in df.columns]

auditoria_extremos = (
    df[cols_auditoria]
    .describe(percentiles=[0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.90, 0.95, 0.98, 0.99, 0.995, 0.998, 0.999])
    .T
    .round(2)
)

auditoria_extremos

,count,mean,std,min,0.1%,0.2%,0.5%,1%,2%,5%,50%,90%,95%,98%,99%,99.5%,99.8%,99.9%,max
trip_seconds,12406255.0,1261.47,929.02,90.00,116.00,120.00,146.00,177.00,205.00,275.00,985.00,2586.00,3178.00,3831.00,4260.00,4623.00,4990.00,5166.00,5400.0
trip_miles,12406255.0,7.25,6.77,0.30,0.30,0.31,0.36,0.40,0.50,0.64,4.22,17.45,18.40,21.77,26.13,28.90,32.60,35.68,50.0
fare,12406255.0,22.20,16.17,3.25,4.00,4.25,4.50,4.50,5.00,5.50,15.75,44.75,49.25,61.10,68.45,75.41,85.19,93.75,150.0
tips,12406255.0,2.76,3.98,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,9.50,10.75,13.25,15.10,17.56,20.40,23.20,30.0
tolls,12406255.0,0.02,0.22,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2.00,3.00,4.00,5.0
extras,12406255.0,1.86,4.82,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,5.00,6.00,21.00,28.95,34.00,40.25,44.00,50.0
trip_total,12406255.0,27.10,21.38,3.25,4.25,4.25,4.50,5.00,5.25,6.25,18.25,58.30,65.00,80.50,92.00,104.70,122.48,133.50,189.5
trip_minutes,12406255.0,21.02,15.48,1.50,1.93,2.00,2.43,2.95,3.42,4.58,16.42,43.10,52.97,63.85,71.00,77.05,83.17,86.10,90.0
trip_hours,12406255.0,0.35,0.26,0.02,0.03,0.03,0.04,0.05,0.06,0.08,0.27,0.72,0.88,1.06,1.18,1.28,1.39,1.44,1.5
speed_mph,12406255.0,18.38,11.21,1.00,1.36,1.65,2.61,3.73,4.71,6.00,14.97,35.00,41.13,46.96,50.41,53.44,57.13,59.78,70.0


### Exporto staging en parquet

In [52]:
# df.to_parquet(DATA_STAGING / "taxi_trips_chicago_staging.parquet", index=False)

### Exportar recorte 2026

In [53]:
#   # ME QUEDO CON EL 2026 PARA TRAGBAJAR EN DASHBOARDS
#   df_2026 = df[df["trip_year"] == 2026].copy()
#   
#   df_2026.to_csv(DATA_STAGING / "taxi_trips_chicago_2026.csv", index=False)